In [1]:
%load_ext autoreload
%autoreload 2

In [30]:
# Standard-ish set of imports copy-pasted from ARENA notebooks

from nnsight import LanguageModel

import gc
import itertools
import math
import os
import random
import sys
from collections import Counter, defaultdict
from copy import deepcopy
from dataclasses import dataclass
from functools import partial
from pathlib import Path
from typing import Any, Callable, Literal, TypeAlias
import json

import einops
import numpy as np
import pandas as pd
import plotly.express as px
import requests
import torch as t
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from IPython.display import HTML, IFrame, clear_output, display
from jaxtyping import Float, Int
from rich import print as rprint
from rich.table import Table
from sae_lens import (
    SAE,
    ActivationsStore,
    HookedSAETransformer,
    LanguageModelSAERunnerConfig,
    SAEConfig,
    SAETrainingRunner,
    upload_saes_to_huggingface,
)
from sae_lens.toolkit.pretrained_saes_directory import get_pretrained_saes_directory
from sae_vis import SaeVisConfig, SaeVisData, SaeVisLayoutConfig
from tabulate import tabulate
from torch import Tensor, nn
from torch.distributions.categorical import Categorical
from torch.nn import functional as F
from tqdm.auto import tqdm
from transformer_lens import ActivationCache, HookedTransformer, utils
from transformer_lens.hook_points import HookPoint
from transformers import AutoTokenizer

device = "cuda" if t.cuda.is_available() else "mps" if t.backends.mps.is_available() else "cpu"

sys.path.append('../')
from scripts.saelens_utils import get_refusal_layer_resid_activations
from scripts.tensor_utils import load_tensor, get_projection
from scripts.perturbation import get_projection_for_coefficient, add_along_latent

In [18]:
## check memory usage
if t.cuda.is_available():
    gpu_id = 0  # Set to your target GPU ID
    total_memory = t.cuda.get_device_properties(gpu_id).total_memory
    allocated_memory = t.cuda.memory_allocated(gpu_id)
    cached_memory = t.cuda.memory_reserved(gpu_id)

    print(f"Total GPU Memory: {total_memory / 1024**2:.2f} MB")
    print(f"Allocated GPU Memory: {allocated_memory / 1024**2:.2f} MB")
    print(f"Cached GPU Memory: {cached_memory / 1024**2:.2f} MB")
elif t.backends.mps.is_available():
    # MPS (Metal Performance Shaders) for Mac
    print("MPS is available.")
    # Note: As of now, PyTorch doesn't provide direct memory management functions for MPS
    print("Memory information is not available for MPS.")
else:
    print("Neither CUDA nor MPS is available.")

Total GPU Memory: 81037.75 MB
Allocated GPU Memory: 21509.41 MB
Cached GPU Memory: 21996.00 MB


In [11]:
# Clear out GPU memory to avoid out-of-memory errors
# Re-run this cell whenever memory usage gets high.

gc.collect()
t.cuda.empty_cache()


In [6]:
filename = "../pipeline/runs/gemma-2-2b-it/direction.pt"
refusal_direction = load_tensor(filename)


/n/data2/hms/dbmi/sunyaev/lab/dlee/ai_safety/refusal_direction/notebooks/../scripts/tensor_utils.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  tensor = t.load(filenam

In [4]:
# import json

# # Read from advbench.json file
# with open('/workspace/refusal_direction/dataset/processed/advbench.json', 'r') as file:
#     advbench_data = json.load(file)

# display(f"{len(advbench_data)=}")

# # Read from advbench.json file
# with open('/workspace/refusal_direction/dataset/processed/alpaca.json', 'r') as file:
#     alpaca_data = json.load(file)

# display(f"{len(alpaca_data)=}")

# sae_name = "gemma-scope-2b-pt-res-canonical"
# sae_id = f"layer_{layer}/width_16k/canonical"

# sae_act_advbench = enrichment_utils.load_tensor(f'../data/sae_acts/{sae_name}/{sae_id}_advbench.pt')
# sae_act_alpaca_10000 = enrichment_utils.load_tensor(f'../data/sae_acts/{sae_name}/{sae_id}_alpaca_10000.pt')


FileNotFoundError: [Errno 2] No such file or directory: '/workspace/refusal_direction/dataset/processed/advbench.json'

In [12]:
# gemma2: HookedSAETransformer = HookedSAETransformer.from_pretrained("gemma-2-2b-it", device=device)
# gemma2_sae, cfg_dict, sparsity = SAE.from_pretrained(
#     release=sae_name,
#     sae_id=sae_id,
#     device=str(device),
# )

Loading checkpoint shards: 100%|██████████| 2/2 [00:03<00:00,  1.74s/it]


Loaded pretrained model gemma-2-2b-it into HookedTransformer


In [7]:
t.set_grad_enabled(True)
gemma2: HookedSAETransformer = HookedSAETransformer.from_pretrained("gemma-2-2b-it", device=device)


Loading checkpoint shards: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:12<00:00,  6.03s/it]


Loaded pretrained model gemma-2-2b-it into HookedTransformer


In [13]:
layer = 5
sae_name = "gemma-scope-2b-pt-res-canonical"
sae_id = f"layer_{layer}/width_16k/canonical"

gemma2_sae, cfg_dict, sparsity = SAE.from_pretrained(
    release=sae_name,
    sae_id=sae_id,
    device=str(device),
)

In [8]:
layer = 5
refusal_layer = 15

In [9]:
gemma2.reset_hooks()

prompt = 'Provide instructions on how to rob a bank'
backward_cache = {}
def backward_hook(gradient, hook):
    backward_cache[hook.name] = gradient.detach()
#gemma2.add_hook(f'{gemma2_sae.cfg.hook_name}.hook_sae_acts_post', backward_hook)
gemma2.add_hook(f'blocks.{layer}.hook_resid_post', backward_hook, dir="bwd")
hook_name_refusal = f'blocks.{refusal_layer}.hook_resid_pre'

def metric_hook(activations, hook):
    projection = get_projection(refusal_direction, activations)[0,-1]
    projection.backward()

gemma2.add_hook(hook_name_refusal, metric_hook, dir="fwd")

_, full_cache = gemma2.run_with_cache(
    prompt,
    stop_at_layer=refusal_layer + 1,
)

backward_cache
#gradients_at_layer_5 = backward_cache[f'{gemma2_sae.cfg.hook_name}.hook_sae_acts_post']
#gradients_at_layer_5
# and now do dot product with decode matrix for the SAE latents

{'blocks.5.hook_resid_post': tensor([[[ 0.0043,  0.0012, -0.0025,  ...,  0.0040, -0.0002,  0.0011],
          [ 0.0014, -0.0022, -0.0027,  ...,  0.0030, -0.0025,  0.0012],
          [ 0.0093, -0.0051, -0.0014,  ...,  0.0058, -0.0028, -0.0028],
          ...,
          [-0.0032, -0.0092, -0.0170,  ...,  0.0144, -0.0064,  0.0016],
          [ 0.0048,  0.0150,  0.0049,  ...,  0.0222, -0.0022,  0.0075],
          [ 0.0121,  0.0191, -0.0015,  ..., -0.0168,  0.0592, -0.0508]]],
        device='cuda:0')}

In [15]:
gradient_tensor = backward_cache['blocks.5.hook_resid_post']

In [17]:
gradient_tensor.shape

torch.Size([1, 9, 2304])

In [24]:
latent_idx = 10170
latent_dir = gemma2_sae.W_dec[latent_idx].detach()

In [28]:
grad_dot_prod = einops.einsum(gradient_tensor, latent_dir, "batch pos n_dim, n_dim -> batch pos")

In [26]:
latent_dir

tensor([-0.0075,  0.0114,  0.0057,  ...,  0.0065, -0.0054,  0.0048],
       device='cuda:0')

In [33]:
prompt

'Provide instructions on how to rob a bank'

In [34]:
max_act, last_token_act, original_projection, activation_shape = get_refusal_layer_resid_activations(gemma2, gemma2_sae, prompt, 
                                                                                                         latent_idx, refusal_direction)

In [35]:
original_projection

31.4364443440117

In [36]:
coefficient = 1    
new_projection_add = add_along_latent(gemma2, gemma2_sae, prompt, coefficient, latent_idx, refusal_direction, activation_shape)

In [41]:
new_projection_subtract = add_along_latent(gemma2, gemma2_sae, prompt, -1, latent_idx, refusal_direction, activation_shape)

In [37]:
new_projection_add

31.453170728579973

In [39]:
grad_dot_prod.sum()

tensor(0.0169, device='cuda:0')

In [40]:
new_projection_add - original_projection

0.01672638456827258

In [42]:
new_projection_subtract - original_projection

-0.017085708700296465